<img src="https://github.com/hernancontigiani/ceia_memorias_especializacion/raw/master/Figures/logoFIUBA.jpg" width="500" align="center">


# Procesamiento de lenguaje natural
## Modelo de lenguaje con tokenización por caracteres

### Consigna
- Seleccionar un corpus de texto sobre el cual entrenar el modelo de lenguaje.
- Realizar el pre-procesamiento adecuado para tokenizar el corpus, estructurar el dataset y separar entre datos de entrenamiento y validación.
- Proponer arquitecturas de redes neuronales basadas en unidades recurrentes para implementar un modelo de lenguaje.
- Con el o los modelos que consideren adecuados, generar nuevas secuencias a partir de secuencias de contexto con las estrategias de greedy search y beam search determístico y estocástico. En este último caso observar el efecto de la temperatura en la generación de secuencias.


### Sugerencias
- Durante el entrenamiento, guiarse por el descenso de la perplejidad en los datos de validación para finalizar el entrenamiento. Para ello se provee un callback.
- Explorar utilizar SimpleRNN (celda de Elman), LSTM y GRU.
- rmsprop es el optimizador recomendado para la buena convergencia. No obstante se pueden explorar otros.


In [1]:
import random
import io
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from tensorflow import keras
from tensorflow.keras import layers
from keras.utils import to_categorical
from keras.models import Sequential
from keras.layers import Dense, LSTM, Embedding, Dropout
from tensorflow.keras.losses import SparseCategoricalCrossentropy

I0000 00:00:1777729316.484504    3773 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1777729316.488339    3773 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1777729316.797685    3773 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1777729318.418456    3773 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.

## DESARROLLO
Para corpus utlizaré una biliografía diferente, también de textos.info.

In [2]:
import urllib.request
import bs4 as bs

### Dataset
Para el desarrollo de este trabajo práctico se utiliza como corpus la novela "Viaje al centro de la Tierra" de Julio Verne.

El texto se obtiene desde el mismo repositorio utilizado por la cátedra (https://www.textos.info).

In [3]:
raw_html = urllib.request.urlopen('https://www.textos.info/julio-verne/viaje-al-centro-de-la-tierra/ebook')
raw_html = raw_html.read()

# Parsear artículo, 'lxml' es el parser a utilizar
article_html = bs.BeautifulSoup(raw_html, 'lxml')

# Encontrar todos los párrafos del HTML (bajo el tag <p>)
# y tenerlos disponible como lista
article_paragraphs = article_html.find_all('p')

article_text = ''

for para in article_paragraphs:
    article_text += para.text + ' '

# pasar todo el texto a minúscula
article_text = article_text.lower()

In [4]:
article_text[:1000]

' el domingo 24 de mayo de 1863, mi tío, el profesor lidenbrock, entró \r\nrápidamente a su hogar, situado en el número 19 de la könig‑strasse, una\r\n de las calles más tradicionales del barrio antiguo de hamburgo. marta, su excelente criada, se preocupó sobremanera, creyendo que se \r\nhabía retrasado, pues apenas empezaba a cocinar la comida en el \r\nhornillo. “bueno” —pensé para mí—, “si mi tío viene con hambre, se va a armar \r\nla de san quintín; porque no conozco a otro hombre de menos paciencia”. —¡tan temprano y ya está aquí el señor lidenbrock! —exclamó la pobre marta, con arrebol, entreabriendo la puerta del comedor. —sí, marta; pero tú no tienes la culpa de que la comida no esté lista\r\n todavía, porque es temprano, aún no son las dos. acaba de dar la media \r\nhora en san miguel. —¿y por qué ha venido tan pronto el señor lidenbrock? —él lo explicará, seguramente. —¡ahí viene! yo me escapo. señor axel, cálmelo usted, por favor. y la excelente marta se retiró presurosa a s

### Elegir el tamaño del contexto

In [5]:

#max_context_size = 50
max_context_size = 100
#max_context_size = 100

In [6]:
from tensorflow.keras.utils import pad_sequences 
chars_vocab = set(article_text)
len(chars_vocab)
char2idx = {k: v for v,k in enumerate(chars_vocab)}
idx2char = {v: k for k,v in char2idx.items()}

###  Tokenizar

In [7]:
tokenized_text = [char2idx[ch] for ch in article_text]

In [8]:
tokenized_text[:1000]

[59,
 47,
 42,
 59,
 49,
 17,
 5,
 65,
 19,
 8,
 17,
 59,
 45,
 6,
 59,
 49,
 47,
 59,
 5,
 46,
 51,
 17,
 59,
 49,
 47,
 59,
 68,
 52,
 36,
 69,
 23,
 59,
 5,
 65,
 59,
 13,
 43,
 17,
 23,
 59,
 47,
 42,
 59,
 53,
 31,
 17,
 25,
 47,
 70,
 17,
 31,
 59,
 42,
 65,
 49,
 47,
 19,
 16,
 31,
 17,
 14,
 0,
 23,
 59,
 47,
 19,
 13,
 31,
 1,
 59,
 77,
 20,
 31,
 62,
 53,
 65,
 49,
 46,
 5,
 47,
 19,
 13,
 47,
 59,
 46,
 59,
 70,
 9,
 59,
 3,
 17,
 8,
 46,
 31,
 23,
 59,
 70,
 65,
 13,
 9,
 46,
 49,
 17,
 59,
 47,
 19,
 59,
 47,
 42,
 59,
 19,
 38,
 5,
 47,
 31,
 17,
 59,
 68,
 15,
 59,
 49,
 47,
 59,
 42,
 46,
 59,
 0,
 21,
 19,
 65,
 8,
 2,
 70,
 13,
 31,
 46,
 70,
 70,
 47,
 23,
 59,
 9,
 19,
 46,
 77,
 20,
 59,
 49,
 47,
 59,
 42,
 46,
 70,
 59,
 14,
 46,
 42,
 42,
 47,
 70,
 59,
 5,
 62,
 70,
 59,
 13,
 31,
 46,
 49,
 65,
 14,
 65,
 17,
 19,
 46,
 42,
 47,
 70,
 59,
 49,
 47,
 42,
 59,
 16,
 46,
 31,
 31,
 65,
 17,
 59,
 46,
 19,
 13,
 65,
 8,
 9,
 17,
 59,
 49,
 47,
 59,
 3,
 46,
 5,
 1

### Organizando y estructurando el dataset

In [24]:
p_val = 0.1
num_val = int(np.ceil(len(tokenized_text)*p_val/max_context_size))
train_text = tokenized_text[:-num_val*max_context_size]
val_text = tokenized_text[-num_val*max_context_size:]
tokenized_sentences_train = [train_text[init:init+max_context_size] for init in range(len(train_text)-max_context_size+1)]
X = np.array(tokenized_sentences_train[:-1])
y = np.array(tokenized_sentences_train[1:])
tokenized_sentences_val = [val_text[init:init+max_context_size] for init in range(len(val_text)-max_context_size+1)]

In [10]:
X.shape

(389572, 100)

In [11]:
X[0,:10]
y[0,:10]

array([47, 42, 59, 49, 17,  5, 65, 19,  8, 17])

In [12]:
vocab_size = len(chars_vocab)

# Definiendo el modelo

- Para la resolución del trabajo se evaluará diferentes arquitecturas de redes neuronales recurrentes, utilizando tres variantes: 
    1. SimpleRNN 
    2. LSTM 
    3. GRU. 
    
    De esta forma se analizará el impacto del tipo de unidad recurrente en la capacidad del modelo para capturar patrones del lenguaje.

- Una vez entrenados los modelos, se seleccionará el que presente mejor desempeño para realizar la generación de texto.

- Sobre este modelo se implementarán distintas estrategias de generación de secuencias:

    1. Greedy search
    2. Beam search determinístico
    3. Beam search estocástico

    En el caso del beam search estocástico, se analizará además el efecto de la temperatura sobre la diversidad del texto generado.

In [25]:
from keras.layers import Input, TimeDistributed, CategoryEncoding, SimpleRNN, Dense
from keras.models import Model, Sequential

In [26]:
def build_model(rnn_layer):
    model = Sequential()

    model.add(TimeDistributed(
        CategoryEncoding(num_tokens=vocab_size, output_mode="one_hot"),
        input_shape=(None,1)
    ))

    model.add(rnn_layer)
    model.add(Dense(vocab_size, activation='softmax'))

    model.compile(
        loss='sparse_categorical_crossentropy',
        optimizer='rmsprop'
    )

    return model

Se utilizará el callback proporcionado por la cátedra:

In [29]:
class PplCallback(keras.callbacks.Callback):

    def __init__(self, val_data, history_ppl, model_name, patience=5):
        self.val_data = val_data
        self.history_ppl = history_ppl
        self.model_name = model_name

        self.target = []
        self.padded = []

        count = 0
        self.info = []
        self.min_score = np.inf
        self.patience_counter = 0
        self.patience = patience

        for seq in self.val_data:

            len_seq = len(seq)
            subseq = [seq[:i] for i in range(1,len_seq)]
            self.target.extend([seq[i] for i in range(1,len_seq)])

            if len(subseq)!=0:
                self.padded.append(
                    pad_sequences(subseq, maxlen=max_context_size, padding='pre')
                )

                self.info.append((count,count+len_seq))
                count += len_seq

        self.padded = np.vstack(self.padded)


    def on_epoch_end(self, epoch, logs=None):

        scores = []
        predictions = self.model.predict(self.padded, verbose=0)

        for start,end in self.info:
            probs = [
                predictions[idx_seq,-1,idx_vocab]
                for idx_seq, idx_vocab in zip(range(start,end),self.target[start:end])
            ]

            scores.append(np.exp(-np.sum(np.log(probs))/(end-start)))

        current_score = np.mean(scores)
        self.history_ppl.append(current_score)

        print(f'\n mean perplexity: {current_score} \n')

        if current_score < self.min_score:
            self.min_score = current_score
            self.model.save(f"{self.model_name}.keras")   # 🔥 ahora usa nombre
            print("Saved new model!")
            self.patience_counter = 0
        else:
            self.patience_counter += 1
            if self.patience_counter == self.patience:
                print("Stopping training...")
                self.model.stop_training = True

### 1. Primer modelo a experimentar: SimpleRNN

In [ ]:
print("Entrenando SimpleRNN...")

model = build_model(
    SimpleRNN(200, return_sequences=True, dropout=0.1, recurrent_dropout=0.1)
)

history_ppl_rnn = []

model.fit(
    X, y,
    epochs=20,
    batch_size=256,
    callbacks=[PplCallback(tokenized_sentences_val, history_ppl_rnn, model_name="rnn")]
)

### 2. Segundo modelo a experimentar: LSTM

In [ ]:
print("Entrenando LSTM...")

model = build_model(
    LSTM(200, return_sequences=True, dropout=0.1, recurrent_dropout=0.1)
)

history_ppl_lstm = []

model.fit(
    X, y,
    epochs=20,
    batch_size=256,
    callbacks=[PplCallback(tokenized_sentences_val, history_ppl_lstm, model_name="lstm")]
)

### 3. Tercer modelo a experimentar: GRU

In [ ]:
print("Entrenando GRU...")

model = build_model(
    GRU(200, return_sequences=True, dropout=0.1, recurrent_dropout=0.1)
)

history_ppl_gru = []

model.fit(
    X, y,
    epochs=20,
    batch_size=256,
    callbacks=[PplCallback(tokenized_sentences_val, history_ppl_gru, model_name="gru")]
)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.lineplot(x=range(len(history_ppl_rnn)), y=history_ppl_rnn, label="RNN")
sns.lineplot(x=range(len(history_ppl_lstm)), y=history_ppl_lstm, label="LSTM")
sns.lineplot(x=range(len(history_ppl_gru)), y=history_ppl_gru, label="GRU")

plt.xlabel("Epoch")
plt.ylabel("Perplexity")
plt.legend()
plt.show()

In [ ]:
# ELEGIMOS EL MEJOR MODELO (EL QUE OBTUVO LA MEJOR PERPLEJIDAD) PARA GENERAR TEXTO NUEVO
model = keras.models.load_model('my_model.keras')

ValueError: File not found: filepath=my_model.keras. Please ensure the file is an accessible `.keras` zip file.


### Predicción del próximo caracter

In [ ]:
# Se puede usar gradio para probar el modelo
# Gradio es una herramienta muy útil para crear interfaces para ensayar modelos
# https://gradio.app/

!pip install -q gradio

In [ ]:
import gradio as gr

def model_response(human_text):

    # Encodeamos
    encoded = [char2idx[ch] for ch in human_text.lower() ]
    # Si tienen distinto largo
    encoded = pad_sequences([encoded], maxlen=max_context_size, padding='pre')

    # Predicción softmax
    y_hat = np.argmax(model.predict(encoded)[0,-1,:])


    # Debemos buscar en el vocabulario el caracter
    # que corresopnde al indice (y_hat) predicho por le modelo
    out_word = ''
    out_word = idx2char[y_hat]

    # Agrego la palabra a la frase predicha
    return human_text + out_word

iface = gr.Interface(
    fn=model_response,
    inputs=["textbox"],
    outputs="text")

iface.launch(debug=True)

### Generación de secuencias

In [ ]:
def generate_seq(model, seed_text, max_length, n_words):
    """
        Exec model sequence prediction

        Args:
            model (keras): modelo entrenado
            seed_text (string): texto de entrada (input_seq)
            max_length (int): máxima longitud de la sequencia de entrada
            n_words (int): números de caracteres a agregar a la sequencia de entrada
        returns:
            output_text (string): sentencia con las "n_words" agregadas
    """
    output_text = seed_text
	# generate a fixed number of words
    for _ in range(n_words):
		# Encodeamos
        encoded = [char2idx[ch] for ch in output_text.lower() ]
		# Si tienen distinto largo
        encoded = pad_sequences([encoded], maxlen=max_length, padding='pre')

		# Predicción softmax
        y_hat = np.argmax(model.predict(encoded,verbose=0)[0,-1,:])
		# Vamos concatenando las predicciones
        out_word = ''

        out_word = idx2char[y_hat]

		# Agrego las palabras a la frase predicha
        output_text += out_word
    return output_text

In [ ]:
input_text='habia una vez'

generate_seq(model, input_text, max_length=max_context_size, n_words=30)

###  Beam search y muestreo aleatorio

In [ ]:
# funcionalidades para hacer encoding y decoding

def encode(text,max_length=max_context_size):

    encoded = [char2idx[ch] for ch in text]
    encoded = pad_sequences([encoded], maxlen=max_length, padding='pre')

    return encoded

def decode(seq):
    return ''.join([idx2char[ch] for ch in seq])

In [ ]:
from scipy.special import softmax

# función que selecciona candidatos para el beam search
def select_candidates(pred,num_beams,vocab_size,history_probs,history_tokens,temp,mode):

  # colectar todas las probabilidades para la siguiente búsqueda
  pred_large = []

  for idx,pp in enumerate(pred):
    pred_large.extend(np.log(pp+1E-10)+history_probs[idx])

  pred_large = np.array(pred_large)

  # criterio de selección
  if mode == 'det':
    idx_select = np.argsort(pred_large)[::-1][:num_beams] # beam search determinista
  elif mode == 'sto':
    idx_select = np.random.choice(np.arange(pred_large.shape[0]), num_beams, p=softmax(pred_large/temp)) # beam search con muestreo aleatorio
  else:
    raise ValueError(f'Wrong selection mode. {mode} was given. det and sto are supported.')

  # traducir a índices de token en el vocabulario
  new_history_tokens = np.concatenate((np.array(history_tokens)[idx_select//vocab_size],
                        np.array([idx_select%vocab_size]).T),
                      axis=1)

  # devolver el producto de las probabilidades (log) y la secuencia de tokens seleccionados
  return pred_large[idx_select.astype(int)], new_history_tokens.astype(int)


def beam_search(model,num_beams,num_words,input,temp=1,mode='det'):

    # first iteration

    # encode
    encoded = encode(input)

    # first prediction
    y_hat = model.predict(encoded,verbose=0)[0,-1,:]

    # get vocabulary size
    vocab_size = y_hat.shape[0]

    # initialize history
    history_probs = [0]*num_beams
    history_tokens = [encoded[0]]*num_beams

    # select num_beams candidates
    history_probs, history_tokens = select_candidates([y_hat],
                                        num_beams,
                                        vocab_size,
                                        history_probs,
                                        history_tokens,
                                        temp,
                                        mode)

    # beam search loop
    for i in range(num_words-1):

      preds = []

      for hist in history_tokens:

        # actualizar secuencia de tokens
        input_update = np.array([hist[i+1:]]).copy()

        # predicción
        y_hat = model.predict(input_update,verbose=0)[0,-1,:]

        preds.append(y_hat)

      history_probs, history_tokens = select_candidates(preds,
                                                        num_beams,
                                                        vocab_size,
                                                        history_probs,
                                                        history_tokens,
                                                        temp,
                                                        mode)

    return history_tokens[:,-(len(input)+num_words):]

In [ ]:
# predicción con beam search
salidas = beam_search(model,num_beams=10,num_words=20,input="habia una vez")

In [ ]:
salidas[0]

In [ ]:
# veamos las salidas
decode(salidas[0])